# MedVision-AI Kaggle launcher

This notebook is intentionally thin. It synchronizes `main`, validates the Kaggle runtime and canonical checkpoints, then launches reproducible Stage 1 or Stage 2 with safe auto-resume. Runtime artifacts live outside the repository at `/kaggle/working/medvision_outputs`.


In [ ]:
# 1. Environment and canonical paths
import os
import platform
import subprocess
import sys
from pathlib import Path

os.environ['KERAS_BACKEND'] = 'tensorflow'
WORKING = Path('/kaggle/working')
REPO = WORKING / 'MedVision-AI'
RUNTIME_OUTPUT = WORKING / 'medvision_outputs'
REPO_URL = 'https://github.com/SwastikPandey1024/MedVision-AI.git'
STAGE1_CHECKPOINT = RUNTIME_OUTPUT / 'checkpoints' / 'densenet121_stage1_best.keras'

print('Python:', platform.python_version(), sys.executable)
print('Repository:', REPO)
print('Runtime artifacts:', RUNTIME_OUTPUT)
print('Canonical Stage 1 checkpoint:', STAGE1_CHECKPOINT)


In [ ]:
# 2. Synchronize a fresh or existing Kaggle checkout; artifacts remain untouched.
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'main', REPO_URL, str(REPO)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO), '--no-deps'], check=True)
commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Verified commit:', commit)


In [ ]:
# 3. Verify TensorFlow/Keras, GPU, dataset attachment, and any runtime checkpoint.
import tensorflow as tf
import keras

sys.path.insert(0, str(REPO / 'src'))
from medvision.data.dataset import find_dataset_root
from medvision.models.trainer import find_valid_resume_checkpoint, validate_stage2_source_checkpoint

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__, '| Keras:', keras.__version__)
print('GPUs:', [gpu.name for gpu in gpus])
if not gpus:
    raise RuntimeError('GPU is required for --mode full; enable a Kaggle GPU accelerator.')

dataset_root = find_dataset_root()
labels = list(dataset_root.glob('*train_labels.csv'))
print('Dataset root:', dataset_root)
if not labels:
    raise FileNotFoundError('Attach the RSNA Pneumonia Detection Challenge dataset to this Kaggle notebook.')

stage1_resume_checkpoint = find_valid_resume_checkpoint(STAGE1_CHECKPOINT, 'densenet121')
if stage1_resume_checkpoint:
    print(f'Stage 1 resume checkpoint: valid ({stage1_resume_checkpoint.path} | {stage1_resume_checkpoint.size_bytes / 2**20:.2f} MB | optimizer steps={stage1_resume_checkpoint.optimizer_iterations})')
else:
    print('Stage 1 resume checkpoint: invalid or missing; Stage 1 will start fresh after preflight.')

try:
    stage2_source_checkpoint = validate_stage2_source_checkpoint(STAGE1_CHECKPOINT, 'densenet121')
    print(f'Stage 2 source checkpoint: model weights valid ({stage2_source_checkpoint.path} | {stage2_source_checkpoint.size_bytes / 2**20:.2f} MB | optimizer steps={stage2_source_checkpoint.optimizer_iterations})')
except Exception as exc:
    print(f'Stage 2 source checkpoint: model weights invalid or unavailable ({exc})')


In [ ]:
# 4. Single reproducible launcher. It performs controlled preflight, auto-resumes
# the canonical valid checkpoint if present, preserves save_best_only, and verifies persistence.
subprocess.run([
    sys.executable, 'scripts/train.py',
    '--mode', 'full',
    '--stage', 'stage1',
    '--epochs', '5',
    '--batch-size', '64',
    '--mixed-precision',
    '--auto-resume',
], cwd=REPO, check=True)


## Stage 2 Fine-Tuning (Optional)

If Stage 1 has completed and you want to proceed with Stage 2 fine-tuning, uncomment and run the cell below. Stage 2 starts from the validated Stage 1 checkpoint, unfreezes the top 20 DenseNet layers, keeps BatchNorm frozen, and performs controlled fine-tuning with a lower learning rate.


In [ ]:
# 4b. OPTIONAL: Stage 2 fine-tuning launcher. Uncomment to run after Stage 1 is complete.
# This launcher:
# - Verifies the Stage 1 checkpoint was saved and is valid
# - Loads the validated Stage 1 checkpoint (no random weights or ImageNet reset)
# - Unfreezes top 20 DenseNet layers for fine-tuning
# - Keeps all BatchNormalization layers frozen to preserve pretrained statistics
# - Saves the Stage 2 checkpoint to: /kaggle/working/medvision_outputs/checkpoints/densenet121_stage2_best.keras
# - Supports auto-resume if run multiple times

# Uncomment the lines below to run Stage 2:
# subprocess.run([
#     sys.executable, 'scripts/train.py',
#     '--mode', 'full',
#     '--stage', 'stage2',
#     '--epochs', '3',
#     '--batch-size', '32',
#     '--mixed-precision',
#     '--auto-resume',
# ], cwd=REPO, check=True)


In [ ]:
# 6. Stage 2 fine-tuning launcher (active)

print("=" * 75)
print("STARTING MEDVISION-AI STAGE 2 FINE-TUNING")
print("=" * 75)

subprocess.run([
    sys.executable,
    'scripts/train.py',
    '--mode', 'full',
    '--stage', 'stage2',
    '--epochs', '3',
    '--batch-size', '32',
    '--mixed-precision',
    '--auto-resume',
], cwd=REPO, check=True)

In [ ]:
from pathlib import Path

stage2_ckpt = Path(
    "/kaggle/working/medvision_outputs/checkpoints/"
    "densenet121_stage2_best.keras"
)

print("=" * 75)
print("STAGE 2 CHECKPOINT VERIFICATION")
print("=" * 75)
print("Exists:", stage2_ckpt.exists())

if stage2_ckpt.exists():
    print("Size MB:", round(stage2_ckpt.stat().st_size / 1024**2, 2))
    print("Path:", stage2_ckpt)
else:
    raise RuntimeError("Stage 2 checkpoint was not created.")

In [ ]:
from pathlib import Path

runtime = Path("/kaggle/working/medvision_outputs")

print("=" * 75)
print("FINAL MEDVISION-AI ARTIFACT INVENTORY")
print("=" * 75)

for p in sorted(runtime.rglob("*")):
    if p.is_file():
        print(
            f"{p.relative_to(runtime)} | "
            f"{p.stat().st_size / 1024**2:.2f} MB"
        )

In [ ]:
# 5. Final artifact inventory and summary.
from medvision.models.trainer import verify_checkpoint_persistence

verify_checkpoint_persistence(str(STAGE1_CHECKPOINT))
for path in sorted(RUNTIME_OUTPUT.rglob('*')):
    if path.is_file():
        print(f'{path.relative_to(RUNTIME_OUTPUT)} | {path.stat().st_size / 2**20:.2f} MB')
print('Kaggle runtime artifacts are session-local. Download or explicitly export the checkpoint and metrics if you need them after a runtime reset.')
